# Behavioral Variation: Haiku, Sonnet, and Opus

Three models run the same code-fix task K=20 times each. We extract two views of each trajectory:

1. **Semantic sequence** — the full action trace interleaving tool names with `model_introspection` (MI) labels, showing *what* the model did and *where it reasoned*.
2. **Intent** — a binary label per step: `acting` (tool call) or `introspecting` (reasoning tokens). Run-length encoding of the intent sequence produces a **program string** — the compressed behavioral fingerprint of the trajectory.

The experiment asks: do different models produce different programs, and does program structure correlate with task success?

In [1]:
import json
from pathlib import Path
from trajectory_utils import semantic_sequence, rle_string

DATA = Path("data")

def load_messages(label: str) -> list[list[dict]]:
    manifest = json.loads((DATA / label / "manifest.json").read_text())
    msgs = []
    for i in range(manifest["K"]):
        traj = json.loads((DATA / label / f"trajectory_{i:02d}.json").read_text())
        msgs.append(traj["messages"])
    return msgs

def load_outcomes(label: str) -> list[float]:
    manifest = json.loads((DATA / label / "manifest.json").read_text())
    return manifest["outcomes"]

def load_trajs(label: str):
    manifest = json.loads((DATA / label / "manifest.json").read_text())
    trajs, outcomes = [], manifest["outcomes"]
    for i in range(manifest["K"]):
        trajs.append(json.loads((DATA / label / f"trajectory_{i:02d}.json").read_text()))
    return trajs, outcomes

def show_chains(label: str):
    all_msgs = load_messages(label)
    outcomes = load_outcomes(label)
    sc = sum(1 for o in outcomes if o == 1.0)
    print(f"{label.upper()} (K={len(all_msgs)}, pass={sc}/{len(all_msgs)})")
    print()
    for i, (msgs, outcome) in enumerate(zip(all_msgs, outcomes)):
        tag = "PASS" if outcome == 1.0 else "FAIL"
        chain = rle_string(semantic_sequence(msgs)).replace("model_introspection", "MI")
        print(f"  {i:>2} {tag}  {chain}")

In [2]:
show_chains("haiku")

HAIKU (K=20, pass=3/20)

   0 FAIL  MI → Read → MI → Bash → MI → Read → MI → Edit×4 → MI → Read → MI → Bash → MI
   1 FAIL  MI → Read → MI → Bash → MI → Read → MI → Bash → MI → Edit → MI → Read → MI → Edit×4 → MI → Read → Bash → MI
   2 FAIL  MI → Read → MI → Bash → MI → Read → MI → Bash → Read → MI → Edit → MI → Bash → MI
   3 FAIL  MI → Read → MI → Bash → MI → Bash → Read → MI → Edit×4 → MI → Read → MI → Bash → MI
   4 FAIL  MI → Read → MI → Bash → MI → Read → MI → Bash → Read → MI → Edit → MI → Bash → MI
   5 FAIL  MI → Read → MI → Bash → MI → Read → MI → Bash → MI → Read → MI → Edit×3 → MI → Edit → MI → Read → MI → Bash → MI
   6 FAIL  MI → Read → MI → Bash → MI → Read → MI → Edit → MI → Bash → MI
   7 FAIL  MI → Read → MI → Bash → MI → Bash → Read → MI → Edit → MI → Bash → MI
   8 FAIL  MI → Read → MI → Bash → MI → Read → MI → Bash → MI → Edit×3 → MI → Edit → MI → Bash×2 → MI
   9 FAIL  MI → Read → MI → Bash → MI → Read → MI → Bash → Read → MI → Edit×3 → MI → Edit → MI → Read → MI

In [3]:
show_chains("sonnet")

SONNET (K=20, pass=13/20)

   0 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI → Read → MI
   1 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI → Read → MI
   2 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI
   3 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI → Read → MI
   4 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI → Read → MI
   5 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI
   6 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI
   7 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI → Read → MI
   8 FAIL  MI → Read → MI → Edit×4 → MI → Bash → MI → Read → MI
   9 FAIL  MI → Read → MI → Edit×4 → MI → Bash → MI → Bash → MI
  10 FAIL  MI → Read → MI → Edit×4 → MI → Bash → MI → Read → MI
  11 FAIL  MI → Read → MI → Edit×4 → MI → Bash → MI → Bash → MI
  12 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI → Read → MI
  13 PASS  MI → Read → MI → Edit×3 → MI → Bash → MI
  14 FAIL  MI → Read → MI → Edit×4 → MI → Bash → MI → Bash → MI
  15 FAIL  MI → Read → MI → Edit×4 → MI → Bash → MI → Read → 

In [4]:
show_chains("opus")

OPUS (K=20, pass=20/20)

   0 PASS  Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
   1 PASS  Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
   2 PASS  Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
   3 PASS  Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
   4 PASS  Read → Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
   5 PASS  Read → Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
   6 PASS  Read → Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
   7 PASS  Read → Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
   8 PASS  Read → Glob → Read → TodoWrite → MI → Edit → TodoWrite → Edit → TodoWrite → Edit → TodoWrite → MI → Read → Bash → TodoWrite → MI
   9 PASS  Read → Glob → Read → MI → TodoWrite → Edit → TodoWrite → Edit → TodoWrite → Edit → TodoWrite → MI → Read → Bash → TodoWrite → MI
  10 PASS  Read → Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
  11 PASS  Read → Glob → Read → MI → Edit×3 → MI → Read → Bash → MI
  12 PASS  Read → Glob → Read → MI → Edit×3

## Observations from the semantic chains

- **Opus** starts with tool calls (`Glob → Read`) — no leading MI. All 20 runs follow a nearly identical chain. 20/20 pass.
- **Sonnet** starts with MI, then acts. Two visible modes: `Edit×3` runs pass, `Edit×4` runs fail. 13/20 pass.
- **Haiku** starts with MI but interleaves Bash and Read throughout — more back-and-forth between acting and reasoning. Chains vary significantly across runs. 3/20 pass.

We now formalize the acting/introspecting alternation using `IntentFluctuationField` to see whether these visual patterns hold up as distinct program families.

## `IntentFluctuationField`

The field maps each trajectory step to a binary intent label — **acting** or **introspecting** — and run-length encodes the sequence into a **program string**. Program families group trajectories that share the same program. The field's `width` metric measures how much the 5-dimensional behavioral vectors vary within each family.

In [5]:
from study_field import IntentFluctuationField

def build_field(label: str) -> IntentFluctuationField:
    field = IntentFluctuationField()
    trajs, outcomes = load_trajs(label)
    for traj, outcome in zip(trajs, outcomes):
        field.add(traj, outcome)
    return field

fields = {label: build_field(label) for label in ["haiku", "sonnet", "opus"]}

In [6]:
# Program families per model — how the act/think alternation partitions behavior
for label, field in fields.items():
    m = field.metrics()
    sc = sum(1 for o in field._outcomes if o == 1.0)
    print(f"{label.upper()} K={field.K}  width={m.width():.2f}  pass={sc}/{field.K}")
    print(f"  {len(field.programs)} distinct program(s):")
    for prog in field.programs:
        fam = field.program_family(prog)
        fm = fam.metrics()
        wins = sum(1 for o in fam._outcomes if o == 1.0)
        short = " → ".join(p[0].upper() for p in prog)  # A/I
        print(f"    {fam.K:>2}× (pass={wins}/{fam.K}, width={fm.width():.2f})  {short}")
    print()

HAIKU K=20  width=8.69  pass=3/20
  5 distinct program(s):
    18× (pass=3/18, width=6.98)  I → A → I → A → I → A → I → A → I → A → I → A → I
     6× (pass=1/6, width=1.89)  I → A → I → A → I → A → I → A → I → A → I → A → I → A → I → A → I
     2× (pass=0/2, width=0.50)  I → A → I → A → I → A → I → A → I → A → I → A → I → A → I → A → I → A → I
    20× (pass=3/20, width=8.69)  I → A → I → A → I → A → I → A → I → A → I
     9× (pass=2/9, width=2.00)  I → A → I → A → I → A → I → A → I → A → I → A → I → A → I

SONNET K=20  width=1.06  pass=13/20
  2 distinct program(s):
    15× (pass=8/15, width=0.66)  I → A → I → A → I → A → I → A → I
    20× (pass=13/20, width=1.06)  I → A → I → A → I → A → I

OPUS K=20  width=3.94  pass=20/20
  1 distinct program(s):
    20× (pass=20/20, width=3.94)  A → I → A → I → A → I



## Program families

| Model | Programs | Act/think cycles | Pass rate | Field width |
|-------|----------|-----------------|-----------|-------------|
| Opus | 1 | 3 | 20/20 | 3.94 |
| Sonnet | 2 | 4–5 | 13/20 | 1.06 |
| Haiku | 5 | 6–10 | 3/20 | 8.69 |

Observations:
- **Program count increases as capability decreases.** Opus produces one program across all 20 runs. Sonnet splits into two. Haiku fragments into five.
- **Cycle count also increases.** Opus alternates 3 times, Haiku up to 10. More act/think alternations correlate with lower pass rates.
- **Opus is the only model that starts by acting** (A → I → ...). Sonnet and Haiku both start by introspecting (I → A → ...).
- **Within-family width varies.** Opus has one program but width=3.94 — same alternation pattern, different tool arguments across runs. Sonnet's dominant family has width=0.66 — its trajectories are tightly clustered in behavior space. Haiku's largest family has width=6.98 — even runs that share a program diverge in their behavioral vectors.

In [7]:
# Horizon × program composition — how the rhythm varies across task states
for label, field in fields.items():
    sc = sum(1 for o in field._outcomes if o == 1.0)
    print(f"{label.upper()} (pass={sc}/{field.K})")
    for state in field.states:
        h = field.horizon(state)
        if h.K == 0:
            continue
        hm = h.metrics()
        h_wins = sum(1 for o in h._outcomes if o == 1.0)
        n_progs = len(h.programs)
        print(f"  horizon({state:>12}): K={h.K:>2}  width={hm.width():>6.2f}  pass={h_wins}/{h.K}  programs={n_progs}")
    print()

HAIKU (pass=3/20)
  horizon(       start): K=20  width=  8.69  pass=3/20  programs=5
  horizon(   diagnosed): K=20  width=  8.69  pass=3/20  programs=5
  horizon(     editing): K=19  width=  8.66  pass=2/19  programs=5
  horizon(complete_fix): K=14  width=  3.16  pass=3/14  programs=4
  horizon(      tested): K=14  width=  3.16  pass=3/14  programs=4

SONNET (pass=13/20)
  horizon(       start): K=20  width=  1.06  pass=13/20  programs=2
  horizon(   diagnosed): K=20  width=  1.06  pass=13/20  programs=2
  horizon(     editing): K=20  width=  1.06  pass=13/20  programs=2
  horizon(complete_fix): K=20  width=  1.06  pass=13/20  programs=2
  horizon(      tested): K=20  width=  1.06  pass=13/20  programs=2

OPUS (pass=20/20)
  horizon(       start): K=20  width=  3.94  pass=20/20  programs=1
  horizon(   diagnosed): K=20  width=  3.94  pass=20/20  programs=1
  horizon(     editing): K=20  width=  3.94  pass=20/20  programs=1
  horizon(complete_fix): K=20  width=  3.94  pass=20/20  progra

## Horizons and program stability

Horizons slice the field by task state (start → diagnosed → editing → complete_fix → tested). We check whether program structure holds as trajectories advance through states.

- **Opus** and **Sonnet**: program count and width are constant across all horizons. All trajectories reach all states.
- **Sonnet** has lower "behavioral variance" (width) compared to opus for the dimensions defined by the field meaning sonnet as a whole behaved more consistently then open (not better; just consistent; ie trajectories had less variance on behavioral dimensions but that didnt reflect any change in outcome space)
- **Haiku**: 6 trajectories never reach `complete_fix` (K drops from 20 to 14), and the width drops from 8.69 to 3.16. Program count drops from 5 to 4. The narrowing is driven by attrition — trajectories that used more divergent programs failed to complete the task.